# 03 — Human-in-the-Loop (HITL) Demo

Demonstrates an agent that **pauses to ask the human for input** at decision points.

The `AskHumanTool` lets the agent:
- Present multiple options with descriptions
- Accept free-form text answers
- Track the full interaction history

In this demo we use `CLIHumanHandler` which reads from `stdin`.  
In production, swap it with the `WebHITLBridge`-backed handler used by the FastAPI server.

**Prerequisites**: `OPENAI_API_KEY` set.

In [1]:

from raavan.core.agents.react_agent import ReActAgent
from raavan.providers.llm.openai.openai_client import OpenAIClient
from raavan.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from raavan.core.memory.unbounded_memory import UnboundedMemory
from raavan.core.context.implementations import UnboundedContext
from raavan.extensions.tools.human_input import AskHumanTool
from raavan.core.console import Console

## Run the HITL agent

When the agent calls `ask_human` you will be prompted in the output cell — type your answer and press **Enter**.

In [2]:

# Notebook demo uses a mock handler that auto-responds so the cell is non-blocking.
# In production / CLI use CLIHumanHandler() which prompts via stdin.

from raavan.extensions.tools.human_input import (
    HumanInputHandler, HumanInputRequest, HumanInputResponse,
)

class NotebookMockHandler(HumanInputHandler):
    """Auto-responds with the first available option (for notebook demos)."""

    async def request_input(self, request: HumanInputRequest) -> HumanInputResponse:
        print(f"\n[HITL] Question: {request.question}")
        if request.options:
            chosen = request.options[0]
            print(f"[HITL] Auto-selecting: {chosen.label}")
            return HumanInputResponse(
                selected_option=chosen,
                text=chosen.label,
                is_freeform=False,
            )
        print("[HITL] No options; returning empty freeform.")
        return HumanInputResponse(text="", is_freeform=True)


async def run():
    handler = NotebookMockHandler()
    ask_tool = AskHumanTool(handler=handler, max_requests_per_run=3)
    client = OpenAIClient(model='gpt-4.1-nano')

    agent = ReActAgent(
        name='hitl-assistant',
        description='An assistant that asks for human input when needed',
        model_client=client,
        tools=[ask_tool, CalculatorTool(), GetCurrentTimeTool()],
        memory=UnboundedMemory(),
        model_context=UnboundedContext(),
        system_instructions=(
            'You are a helpful AI assistant. When you need the user\'s preference or '
            'confirmation, use the ask_human tool to present 2–3 options. '
            'You can ask up to 3 questions per conversation.'
        ),
        max_iterations=10,
    )

    result = await Console(agent).run('Help me plan a team dinner for 8 people this Friday.')

    history = ask_tool.interaction_history
    if history:
        print(f'\n--- Human Interactions ({len(history)}) ---')
        for h in history:
            print(f'  Q: {h["question"]}')
            print(f'  A: {h["answer"]} (freeform={h["is_freeform"]})')

await run()


You → Help me plan a team dinner for 8 people this Friday.

╭──────────────────────────────────────────────── hitl-assistant ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Let's start by figuring out some details. Would you prefer to dine at a restaurant, host a dinner at           │
│  someone's home, or order in from a special place?                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

completed · 1 steps · 0 tool calls · 349 tokens · 2.0s